# Entrenamiento con MLflow Tracking — resistencia del concreto

Igual que `02_training.ipynb` pero con tracking completo en **Databricks MLflow**:
parámetros, métricas y el modelo quedan registrados en el experimento remoto.

Las credenciales (`DATABRICKS_HOST`, `DATABRICKS_TOKEN`) se leen del archivo `.env`
que ya existe en la raíz del repositorio.

## 1. Importar librerías

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 2. Configurar MLflow → Databricks

`load_dotenv` lee el `.env` de la raíz y carga `DATABRICKS_HOST` y `DATABRICKS_TOKEN`
en `os.environ`. Con eso MLflow sabe a qué workspace conectarse y cómo autenticarse.

In [ ]:
from dotenv import load_dotenv

# Carga .env desde la raíz del repo (un nivel arriba del notebook)
load_dotenv("../.env")

DATABRICKS_HOST  = os.environ["DATABRICKS_HOST"]
DATABRICKS_TOKEN = os.environ["DATABRICKS_TOKEN"]

# Le indicamos a MLflow que use el workspace de Databricks como backend
mlflow.set_tracking_uri("databricks")

# Nombre del experimento en Databricks (se crea si no existe)
EXPERIMENT_NAME = "/Users/nalayo@itmeet.org/concrete-strength"
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"Tracking URI : {mlflow.get_tracking_uri()}")
print(f"Experimento  : {EXPERIMENT_NAME}")

## 3. Cargar datos crudos

In [ ]:
df = pd.read_csv("../data/raw/concrete_data.csv")

print(f"Forma: {df.shape}")
df.head()

## 4. Limpieza de datos

In [ ]:
df = df.drop_duplicates().dropna()

print(f"Forma tras limpieza: {df.shape}")

## 5. Feature Engineering

In [ ]:
df["water_cement_ratio"] = df["water"] / (df["cement"] + 1e-9)
df["binder_total"] = df["cement"] + df["blast_furnace_slag"] + df["fly_ash"]
df["aggregate_total"] = df["coarse_aggregate"] + df["fine_aggregate"]

print(f"Columnas: {list(df.columns)}")

## 6. Preparar features y target

In [ ]:
TARGET = "concrete_compressive_strength"
TEST_SIZE   = 0.2
RANDOM_STATE = 42

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

## 7. Entrenar y registrar con MLflow

Todo lo que ocurra dentro del bloque `with mlflow.start_run()` queda guardado
en el experimento de Databricks: parámetros, métricas y el artefacto del modelo.

In [ ]:
# Hiperparámetros — cámbialos aquí para comparar runs en Databricks
N_ESTIMATORS = 100
MAX_DEPTH    = None   # None = sin límite

with mlflow.start_run(run_name="random_forest_baseline"):

    # -- 1. Log de parámetros ---------------------------------------------------
    mlflow.log_params({
        "n_estimators"  : N_ESTIMATORS,
        "max_depth"     : str(MAX_DEPTH),
        "test_size"     : TEST_SIZE,
        "random_state"  : RANDOM_STATE
    })

    # -- 2. Entrenar ------------------------------------------------------------
    model = RandomForestRegressor(
        n_estimators=N_ESTIMATORS,
        max_depth=MAX_DEPTH,
        random_state=RANDOM_STATE,
    )
    model.fit(X_train, y_train)

    # -- 3. Métricas ------------------------------------------------------------
    y_pred = model.predict(X_test)
    mae   = mean_absolute_error(y_test, y_pred)
    rmse  = np.sqrt(mean_squared_error(y_test, y_pred))
    r2    = r2_score(y_test, y_pred)

    mlflow.log_metrics({"mae": mae, "rmse": rmse, "r2": r2})

    # -- 4. Artefacto: modelo --------------------------------------------------
    X_train_f64 = X_train.astype("float64")
    signature   = mlflow.models.infer_signature(X_train_f64, model.predict(X_train))
    mlflow.sklearn.log_model(
        sk_model=model,
        name="random_forest",
        signature=signature,
        input_example=X_train_f64.head(5),   # 5 filas representativas
    )

    # -- 5. Gráfico predicho vs real (artefacto + display local) ---------------
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(y_test, y_pred, alpha=0.6, edgecolors="k", linewidths=0.4)
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    ax.plot(lims, lims, "r--", linewidth=1.5, label="Predicción perfecta")
    ax.set_xlabel("Valor real (MPa)")
    ax.set_ylabel("Predicción (MPa)")
    ax.set_title(f"Predicho vs Real  |  R²={r2:.3f}")
    ax.legend()
    plt.tight_layout()

    mlflow.log_figure(fig, artifact_file="predicho_vs_real.png")
    plt.show()

    run_id = mlflow.active_run().info.run_id
    print(f"Run registrado  : {run_id}")
    print(f"MAE  : {mae:.3f}")
    print(f"RMSE : {rmse:.3f}")
    print(f"R²   : {r2:.3f}")